# Landesvermessung

Die Landesvermessung basiert traditionellerweise im Wesentlichen auf der
Anwendung von trigonometrischen Funktionen. Man spricht daher auch von
Triangulation. 

::: {.callout-note title="Trigonometrische Grundlagen"}

In einem allgemeinen Dreieck (vgl. @fig-dreieck) gelten die folgenden
beiden Sätze:

![Allgemeines Dreieck.](dreieck.svg){#fig-dreieck}

**Sinussatz**

$$
\frac{a}{\sin \alpha} = \frac{b}{\sin \beta} = \frac{c}{\sin \gamma}
$$

**Kosinussatz**

\begin{align*}
a^2 &= b^2 + c^2 - 2 \cdot b \cdot c \cdot \cos \alpha \\
b^2 &= c^2 + a^2 - 2 \cdot c \cdot a \cdot \cos \beta \\
c^2 &= a^2 + b^2 - 2 \cdot a \cdot b \cdot \cos \gamma
\end{align*}

:::

## Anwendungsübung

Im Folgenden soll das anhand der Vermessung einer Strecke in
unzugänglichem Gelände illustriert und angewendet werden.

Gegeben sei die folgende Ausgangslage (vgl. @fig-vermessung):

![Vermessungsskizze[@doberkatEbeneTrigonometrieAnalytische2024, S. 56].](vermessung.svg){#fig-vermessung}

Gesucht ist die Strecke $\overline{AB}$ ($\ell$). Gemessen werden kann die
Strecke $\overline{CD}$ ($s$) sowie die Winkel $\alpha_1$, $\alpha_2$,
$\beta_1$ und $\beta_2$.

### Arbeitsschritte 

Mit den gemessenen Grössen lassen sich der Reihe nach die folgenden
fehlenden Grössen berechnen:

1. $\gamma_1 = 180^{\circ} - (\alpha_1 + \beta_1)$
2. Mit Hilfe des Sinussatzes $a = \frac{s \cdot \sin \alpha_1}{\sin \gamma_1}$
3. $\gamma_2 = 180^{\circ} - (\alpha_2 + \beta_2)$
4. Mit Hilfe des Sinussatzes $b = \frac{s \cdot \sin \alpha_2}{\sin \gamma_2}$
5. Mit Hilfe des Kosinussatzes $\ell =\sqrt{a^2 + b^2 - 2ab \cos (\beta_2 - \beta_1)}$

### Berechnungen mit Hilfe von Python

Es ist deutlich einfacher, alle Messwerte einer Python-Funktion zu
übergeben, als alle Schritte manuell mit Hilfe eines Taschenrechners
einzeln zu berechnen. Dabei ist allerdings zu beachten, dass Python
Winkelberechnungen in Bogenmass und nicht in Grad vornimmt.

::: {.callout-note}

## Winkel in Bogenmass

:::: {.columns}

::: {.column width="50%"}

![](circle.svg)
:::

::: {.column width="50%"}
Das Bogenmass $\widehat{\varphi}$ eines Winkels ist das Verhältnis der
Länge des vom Winkel $\varphi$ ausgeschnittenen Kreisbogens zum Radius
des Kreises. Es wird auch mit $\operatorname{arc} \varphi$ bezeichnet.

$$\widehat{\varphi} = \frac{b}{r} = \operatorname{arc} \varphi =
\frac{\pi}{180^\circ} \varphi \iff \varphi = \frac{180^\circ}{\pi}
\widehat{\varphi}$$ 

Auf dem Einheitskreis ($r=1$) gilt: $\widehat{\varphi} = b$Die Einheit
des Bogenmasses ist der Radiant (rad). Ein Winkel von 1 rad entspricht
einem Kreisbogen, der dieselbe Länge wie der Radius
hat.[@durandiFormelnTabellenBegriffe2022, S. 91]  
:::
:::
:::

Glücklicherweise stellt Python im Modul `math` die Funktion
`math.radians()` zur Verfügung, mit welcher Grad in Bogenmass
umgerechnet werden können. Hierzu muss das Modul vorab mit `import math`
importiert werden. Das Modul `math` stellt darüber hinaus auch alle
erforderlichen trigonometrischen Funktionen (`math.sin()` und
`math.cos()`) zur Verfügung.

Um die Python-Funktion so schlank wie möglich und gut lesbar zu
halten, wird als erstes eine Hilfsfunktion implementiert, welche Grad in
Bogenmass umrechnet. Weil es aber auch Kompasse mit einer Eichung in
A‰ und Gon gibt, soll die Funktion alternativ auch Werte
in diesen Einheiten umrechnen.


In [1]:
import math

In [ ]:
def angleconverter(angle: float, a: bool = False, gon: bool = False) -> float:
    """Konvertiert einen Winkel in das Bogenmass (Radiant).

    Unterstützt die Umrechnung von Altgrad (Standard), Artillerie-Promille
    oder Gon. Es darf jeweils nur ein Modus aktiv sein.

    Args:
        angle (float): Der zu konvertierende Winkelwert.
        a (bool, optional): Wenn True, wird der Winkel als Artillerie-Promille
            (A‰) interpretiert. Standard ist False.
        gon (bool, optional): Wenn True, wird der Winkel als Gon (Neugrad)
            interpretiert. Standard ist False.

    Returns:
        float: Der umgerechnete Winkel im Bogenmass.

    Raises:
        ValueError: Wenn sowohl 'a' als auch 'gon' auf True gesetzt sind.
    """
    if a and gon:
        raise ValueError(
            "Es kann nur 'a=True' (Artillerie) ODER 'gon=True' gewählt werden."
        )
        
    if a:
        return angle * (math.pi / 3200)
    elif gon:
        return angle * (math.pi / 200)
    else:
        return math.radians(angle)

Wie die für die Berechnung erforderlichen Winkel gemessen werden, soll
am Beispiel von $\alpha_1$ in @fig-vermessung gezeigt werden. Dazu wird
mit dem Kompass von der Position $D$ aus der Azimut nach $B$ und nach
$C$ gemessen. Die Differenz der beiden Messungen entspricht $\alpha_1$.

Um diese Berechnung in einer Python-Funktion zu implementieren, muss sie
als Algorithmus beschrieben werden. Dabei gilt es den Fall zu
berücksichtigen, dass die beiden Messungen die Nordrichtung
einschliessen. Um auch diese Situation abzudecken, bedient man sich der
*modularen Subtraktion*. 

::: {.callout-note}

## Modulare Subtraktion

In der modularen Arithmetik geht es darum, das uns bekannte Rechnen mit
Zahlen aus einer unendlichen Menge auf endliche Zahlenmengen zu
übertragen[@iwanowskiDiskreteMathematikMit2021, S. 166]. So kann
sichergestellt werden, dass das Resultat der Winkelberechnung innerhalb
der Windrose bleibt.

**Allgemein für 360°** $(a - b) \mod 360$

1. Differenz berechnen: $d = a - b$
2. Modulo bilden: $r = d \mod 360$
3. Falls $r$ negativ ist, einmal 360 addieren: $r+360$

**Zahlenbeispiel** $40^{\circ} - 110^{\circ}$

1. $40 - 110 = - 70$
2. $-70 < 0: \text{Wahr}$
3. $−70 + 360 = 290$
4. $40 − 110 \equiv 290 (\mod 360)$

:::